In [ ]:
# IMPORTS

# 1. Symbolic Mathematics (Analytical Derivations & Units)
import sympy as sp
from IPython.display import display, Math # For rendering LaTeX beautifully in Jupyter

# 2. Numerical Arrays and Matrices
import numpy as np

# 3. Scientific Computing (Integration and Eigenvalue Solvers)
from scipy.integrate import quad
from scipy.linalg import eigh

# 4. Visualization (Convergence and Error Graphs)
import matplotlib.pyplot as plt

# 5. Numerical integration methods (custom implementations)
from numerical_integration_methods import integral_solve_using_trapezoidal_rule, integral_solve_using_romberg_method, integral_solve_using_simpsons_rule, scipy_quad_wrapper

In [ ]:
# CONFIGURATION, CONSTANTS & FLAGS

# --- Physical Parameters ---
# We parameterize the physical constants so the equations remain exact.
# They default to 1.0 (Natural Units) to prevent floating-point underflow.
L = 10.0      # The domain size of the infinite well [-L/2, L/2]
HBAR = 1.0    # Reduced Planck's constant
MASS = 1.0    # Particle mass
OMEGA = 1.0   # Oscillator frequency

# --- Variational Parameters ---
N_BASIS = 50  # The size of the basis (k = 1, 2, ..., N_BASIS)
N_STATES = 5  # How many low-lying energy levels to track (should be << N_BASIS)

# --- Execution Flags ---
# These control the logic flow, making debugging much easier.
DEBUG_PRINT = True       # Set to True to print matrix values (keep N_BASIS small if True!)
PLOT_ERROR = True        # Flag to trigger the matplotlib convergence graph
PLOT_WAVEFUNCTION = True # Flag to trigger visualizing the final wavefunctions

# --- Numerical Methods ---
integration_method = 'trapezoidal' # Options: 'analytical', 'scipy', 'trapezoidal', 'romberg', 'simpsons'
eigenvalue_method = 'scipy'        # Options: 'scipy', 'QR', 'Jacobi'
N_GRID = 1000                      # Number of grid intervals for custom integration methods

# The "Strategy" Dispatcher
INTEGRATION_DISPATCH = {
    'trapezoidal': integral_solve_using_trapezoidal_rule,
    'romberg': integral_solve_using_romberg_method,
    'simpsons': integral_solve_using_simpsons_rule,
    'scipy': scipy_quad_wrapper
}

In [ ]:
def basis_function(x, n, L) -> float:
    """
    Evaluates the n-th infinite square well basis function at position x.

    Parameters
    ----------
    x : float
        Position where the function is evaluated.
    n : int
        Basis index.
    L : float
        Width of the well.

    Returns
    -------
    float
        Value of the basis function at x.
    """
    normalization = np.sqrt(2.0 / L)
    argument = (n * np.pi * (x + L / 2.0)) / L
    
    return normalization * np.sin(argument)

def integrand_V(x, n, m, L, mass, omega) -> float:
    """
    The potential energy integrand: chi_n(x) * V(x) * chi_m(x).
    """
    chi_n = basis_function(x, n, L)
    chi_m = basis_function(x, m, L)
    
    V_x = 0.5 * mass * ((omega * x)**2)  # QHO Potential: 1/2 * mass * omega^2 * x^2
    
    return chi_n * V_x * chi_m

In [ ]:
# Calculate the Hamiltonian matrix elements (solve many integrals)

H = np.zeros((N_BASIS, N_BASIS))

# Hamiltonian is hermitian, so H_i,j = H_j,i*. We only need to calculate N_BASIS + (N_BASIS - 1) + (N_BASIS - 2) + ... + 1 = N_BASIS * (N_BASIS + 1) / 2 unique elements. The rest can be filled in by symmetry.

# H = T + V, where T is the kinetic energy term and V is the potential energy term.
for i in range(N_BASIS):
    for j in range(i, N_BASIS):

        # shift indices to match basis functions (important!)
        n = i + 1
        m = j + 1

        # --- Kinetic Energy (Exact analytical result) ---
        # Using T = (hbar^2 * k^2) / 2m
        if n == m:
            k = (n * np.pi) / L
            T = ((HBAR * k)**2) / (2.0 * MASS)
        else:
            T = 0.0
        
        # --- Potential Energy (Numerical Integration) ---
        V = 0.0

        if integration_method == 'analytical':
            pass #TODO
            # Use the analytical formula derived using SymPy
            # V = calculate_V_analytical(n, m, L) #TODO

        elif integration_method in INTEGRATION_DISPATCH:
            integration_method_func = INTEGRATION_DISPATCH[integration_method]

            V = integration_method_func(integrand_V, -L/2, L/2, N_GRID, n, m, L, MASS, OMEGA)
        else:
            raise ValueError("Invalid integration method selected!")
        
        H[i, j] = T + V
        if i != j:
            H[j, i] = H[i, j]  # Fill in the symmetric element (base functions are real, so H is symmetric)

In [ ]:
# Maybe calculate S matrix here for generality, but for the infinite square well, the basis functions are orthonormal, so S = I (identity matrix).

In [ ]:
# Solve secular equation, which is just the eigenvalue problem for the Hamiltonian matrix H.

if eigenvalue_method == 'scipy':
    # Use SciPy's eigh function to solve the eigenvalue problem
    energies, wavefunctions = eigh(H)

elif eigenvalue_method == 'QR':
    # Use a custom QR algorithm to solve the eigenvalue problem
    energies, wavefunctions = custom_qr_algorithm(H)

elif eigenvalue_method == 'Jacobi':
    # Use a custom Jacobi iteration to solve the eigenvalue problem
    energies, wavefunctions = custom_jacobi_iteration(H)

else:
    raise ValueError("Invalid eigenvalue method selected!")

In [ ]:
# The exact theoretical energies (see derivation in the notebook)
exact_energies = np.array([n + 0.5 for n in range(N_STATES)])

# the error between the computed energies and the exact energies
errors = np.abs(energies[:N_STATES] - exact_energies)